In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Fixer l'aléatoire pour avoir exactement les mêmes données si on relance
np.random.seed(42)
random.seed(42)

# 1. Catalogue produits (Secteur Beauté / Luxe)
catalogue = [
    ("Sérum Éclat Intense 30ml", "Soins Visage", 85.0),
    ("Crème Hydratante Velours", "Soins Visage", 68.0),
    ("Huile Démaquillante Douce", "Soins Visage", 34.0),
    ("Contour des Yeux Régénérant", "Soins Visage", 55.0),
    ("Eau de Parfum Nuit d'Ambre 50ml", "Parfums", 115.0),
    ("Eau de Parfum Rose Sauvage 100ml", "Parfums", 145.0),
    ("Brume Parfumée Corps & Cheveux", "Parfums", 48.0),
    ("Rouge à Lèvres Satiné - Carmin", "Maquillage", 38.0),
    ("Mascara Volume Absolu", "Maquillage", 32.0),
    ("Poudre Lumière Sublimatrice", "Maquillage", 52.0),
    ("Coffret Découverte Rituel Beauté", "Coffrets", 160.0),
    ("Coffret Duo Parfums Signature", "Coffrets", 195.0)
]

# 2. Création de 2 500 clients avec des profils d'achat différents
n_customers = 2500
customer_ids = [f"CUST_{10000 + i}" for i in range(n_customers)]
countries = ["France", "Belgique", "Suisse", "Luxembourg", "Canada"]
country_weights = [0.65, 0.12, 0.10, 0.05, 0.08]
customer_country = {cid: np.random.choice(countries, p=country_weights) for cid in customer_ids}

# 3. Génération de 12 000 lignes de commandes entre 2024 et 2026
rows = []
start_date = datetime(2024, 1, 1)
end_date = datetime(2026, 10, 31)
days_range = (end_date - start_date).days

for i in range(12000):
    order_id = f"ORD_{50000 + i}"
    # Certains clients achètent beaucoup plus que d'autres (effet Pareto)
    cid = np.random.choice(customer_ids, p=np.random.dirichlet(np.ones(n_customers) * 0.6))
    
    # Date avec saisonnalité (plus d'achats en novembre/décembre et février)
    random_days = random.randint(0, days_range)
    order_date = start_date + timedelta(days=random_days)
    if order_date.month not in [2, 11, 12] and random.random() < 0.35:
        order_date = order_date.replace(month=random.choice([2, 11, 12]), day=random.randint(1, 28))
        
    prod, cat, price = random.choice(catalogue)
    qty = np.random.choice([1, 2, 3, 4], p=[0.65, 0.22, 0.09, 0.04])
    country = customer_country[cid]
    
    rows.append([cid, order_id, order_date.strftime("%Y-%m-%d"), prod, cat, price, qty, country])

df = pd.DataFrame(rows, columns=["customer_id", "order_id", "date", "product", "category", "price", "quantity", "country"])

# 4. Injection volontaire d'anomalies réalistes (pour l'étape de nettoyage !)
# a) Retours produits (quantités négatives sur 3% des commandes)
return_idx = df.sample(frac=0.03, random_state=42).index
df.loc[return_idx, "quantity"] = -df.loc[return_idx, "quantity"]

# b) Valeurs manquantes sur customer_id et country
df.loc[df.sample(frac=0.015, random_state=1).index, "customer_id"] = np.nan
df.loc[df.sample(frac=0.01, random_state=2).index, "country"] = np.nan

# c) Fautes de frappe sur les pays et espaces inutiles
typo_idx = df.sample(frac=0.02, random_state=3).index
df.loc[typo_idx, "country"] = df.loc[typo_idx, "country"].replace({"France": "france ", "Belgique": "BE", "Suisse": "suisse"})

# d) Formats de dates incohérents (DD/MM/YYYY mélangé avec YYYY-MM-DD)
date_idx = df.sample(frac=0.05, random_state=4).index
df.loc[date_idx, "date"] = pd.to_datetime(df.loc[date_idx, "date"]).dt.strftime("%d/%m/%Y")

# e) Ajout de 150 lignes en doublon parfait
duplicates = df.sample(n=150, random_state=5)
df = pd.concat([df, duplicates], ignore_index=True)

# Sauvegarde dans le dossier data
df.to_csv("../data/raw_ecommerce_sales.csv", index=False)
print(f"Fichier généré avec succès ! Nombre total de lignes : {len(df)}")
df.head(10)

ImportError: numpy._core.multiarray failed to import